In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2
# sys.path.append('/home/wolfgang/repos/sleep_research_io')
# from sleep_research_functions import *
%matplotlib widget
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from icu_sleep_breathing_2020_help_functions import * 

pd.options.display.max_rows = 300
pd.options.display.max_columns = 300

font = {'family' : 'normal',
        'weight' : 'normal',
        'size'   : 7}

matplotlib.rc('font', **font)

### load summary tables:

In [2]:
plots_savedir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/plots'

In [3]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria(sleeplab_matched=True)

# of subjects ICU before inclusion criteria: 108
# of 12-hour segments ICU before inclusion criteria: 1230
# of subjects ICU after inclusion criteria: 103
# of 12-hour segments ICU after inclusion criteria: 621
24-hour segments: 256
12-hour segments: 365

# of subjects sleeplab before inclusion criteria: 307
# of 12-hour segments sleeplab before inclusion criteria: 307
# of subjects sleeplab after inclusion criteria: 307
# of 12-hour segments sleeplab after before inclusion criteria: 307


In [4]:
for modality in ['breathing', 'ecg_nn']:
    var_n2 = 'stages_distribution_MODALITY_N2'.replace('MODALITY', modality)
    var_n3 = 'stages_distribution_MODALITY_N3'.replace('MODALITY', modality)
    var_sum = 'stages_distribution_MODALITY_N2N3'.replace('MODALITY', modality)
    
    summary_days_icu[var_sum] = summary_days_icu[var_n2] + summary_days_icu[var_n3]
    summary_days_sleeplab[var_sum] = summary_days_sleeplab[var_n2] + summary_days_sleeplab[var_n3]


In [7]:
cpc_vars = ['cpc_hfc_lfc_ratio', 'cpc_hfc', 'cpc_lfc']

for var in cpc_vars:

    summary_days_icu[var + '_iqr'] = summary_days_icu[var + '_q75'] - summary_days_icu[var + '_q25']
    summary_days_sleeplab[var + '_iqr'] = summary_days_sleeplab[var + '_q75'] - summary_days_sleeplab[var + '_q25']

In [8]:
summary_days_icu.loc[:, ['stages_distribution' in x for x in summary_days_icu.columns]] *= 100
summary_days_sleeplab.loc[:, ['stages_distribution' in x for x in summary_days_sleeplab.columns]] *= 100

summary_f_icu = summary_days_icu.loc[summary_days_icu.day_cat == 'f', :]
summary_dn_icu = summary_days_icu.loc[summary_days_icu.day_cat != 'f', :]

summary_days_sleeplab_full = summary_days_sleeplab.copy()

In [9]:
### 3x2 plot
# cols: data type, rows: stage version (resp, ecg, expert)

variables_template = ['sleep_hours_MODALITY',
             'stages_distribution_MODALITY_S',
             'stages_distribution_MODALITY_R',
             'stages_distribution_MODALITY_N1',
            'stages_distribution_MODALITY_N2',
             'stages_distribution_MODALITY_N3',
             'sleep_MODALITY_sfi',
             'sleep_MODALITY_sfi_w',
             'sleep_MODALITY_arousali',
            ]
x_labels = ['Sleep Dur. (h)', 'Sleep Efficiency', 'Stage R (%)', 'Stage N1 (%)', 'Stage N2 (%)', 'Stage N3 (%)', 'SFI', 'SFI Wake', 'Arousal I.']

In [10]:
Xy_sleeplab = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_all'] == 1].copy()

summary_days_icu_n = summary_days_icu.loc[summary_days_icu.day_cat == 'n', :]

summary_days_sleeplab = summary_days_sleeplab_full.loc[summary_days_sleeplab_full.matched_all == 1].copy()

min_hours_sleep_icu = 4
icu_4 = summary_dn_icu.loc[np.any([summary_dn_icu.sleep_hours_breathing >= min_hours_sleep_icu, 
                                               summary_dn_icu.sleep_hours_ecg_nn >= min_hours_sleep_icu],
                                               axis=0), :]

icu_4_strict = summary_dn_icu.loc[np.all([summary_dn_icu.sleep_hours_breathing >= min_hours_sleep_icu, 
                                               summary_dn_icu.sleep_hours_ecg_nn >= min_hours_sleep_icu],
                                               axis=0), :]


icu_version = 'summary_days_icu_n'

if icu_version == 'icu_4_strict':
    Xy_icu = icu_4_strict.copy()
    suptitle = f'ICU data with >4 hour sleep prediction for both models.\nn_sleeplab = {Xy_sleeplab.shape[0]} n_icu = {Xy_icu.shape[0]}'
elif icu_version == 'icu_4':
    Xy_icu = icu_4.copy()
    suptitle = f'ICU data with >4 hour sleep prediction for any of the two models.\nn_sleeplab = {Xy_sleeplab.shape[0]} n_icu = {Xy_icu.shape[0]}'
elif icu_version == 'summary_days_icu_n':
    Xy_icu = summary_days_icu_n
    suptitle = f'ICU data with all night segments.\nn_sleeplab = {Xy_sleeplab.shape[0]} n_icu = {Xy_icu.shape[0]}'
elif icu_version == 'summary_dn_icu':
    Xy_icu = summary_dn_icu
    suptitle = f'ICU data with all 12-hour segments.\nn_sleeplab = {Xy_sleeplab.shape[0]} n_icu = {Xy_icu.shape[0]}'

print(f'ICU Version = {icu_version}')
print(suptitle)

ICU Version = summary_days_icu_n
ICU data with all night segments.
n_sleeplab = 220 n_icu = 179


In [11]:
Xy_icu.shape

(179, 210)

In [12]:
[x for x in Xy_icu.columns if 'cpc' in x]

['cpc_hfc_lfc_ratio_mean',
 'cpc_hfc_lfc_ratio_std',
 'cpc_hfc_lfc_ratio_median',
 'cpc_hfc_lfc_ratio_q25',
 'cpc_hfc_lfc_ratio_q75',
 'cpc_hfc_mean',
 'cpc_hfc_std',
 'cpc_hfc_median',
 'cpc_hfc_q25',
 'cpc_hfc_q75',
 'cpc_hfc_sum',
 'cpc_lfc_mean',
 'cpc_lfc_std',
 'cpc_lfc_median',
 'cpc_lfc_q25',
 'cpc_lfc_q75',
 'cpc_lfc_sum',
 'cpc_hfc_lfc_ratio_iqr',
 'cpc_hfc_iqr',
 'cpc_lfc_iqr']

In [13]:
def default_scatter():
    plt.figure(figsize=figsize)
    plt.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c='goldenrod', s=s, alpha=alpha) # orange
    plt.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(['Sleeplab', 'ICU'])
    plt.title(title)
    plt.tight_layout()
    
    
def default_scatter_ax0(ax):
    ax.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c='goldenrod', s=s, alpha=alpha)
    ax.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(['Sleeplab', 'ICU'])
    ax.set_title(title)
    
def default_scatter_ax(ax, x_var, y_var,title):
    ax.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c='goldenrod', s=s, alpha=alpha)
    ax.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
#     ax.legend(['Sleeplab', 'ICU'])
    ax.set_title(title)
    


In [14]:
vars_to_plot = cpc_vars
units = ['','','']
titles = ['CPC HFC/LFC', 'CPC HFC', 'CPC LFC']

fig, ax = plt.subplots(1, 3, figsize = (6, 3))
ax = ax.flatten()
s=6
alpha = 0.5

for i_var, var_tmp in enumerate(vars_to_plot):
    
    xlabel = 'Median' + units[i_var]
    ylabel = 'IQR' + units[i_var]

    x_var = var_tmp + '_median'
    y_var = var_tmp + '_iqr'
    title = titles[i_var]
    
    default_scatter_ax(ax[i_var], x_var, y_var, title)
    
ax[0].legend(['Sleeplab', 'ICU'])
plt.tight_layout()

# plt.savefig(os.path.join(plots_savedir, f'Figure_CPC.jpg'), dpi=500)
# plt.savefig(os.path.join(plots_savedir, f'Figure_CPC.svg'))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

findfont: Font family ['normal'] not found. Falling back to DejaVu Sans.
findfont: Font family ['normal'] not found. Falling back to DejaVu Sans.


In [15]:
vars_to_plot = ['cpc_hfc_lfc_ratio']
units = ['', '', '']
titles = ['CPC HFC/LFC']

fig, ax = plt.subplots(1, 1, figsize = (5, 5))
# ax = ax.flatten()
s=6
alpha = 0.5

for i_var, var_tmp in enumerate(vars_to_plot):
    
    xlabel = 'Median' + units[i_var]
    ylabel = 'IQR' + units[i_var]

    x_var = var_tmp + '_median'
    y_var = var_tmp + '_iqr'
    title = titles[i_var]
    
    default_scatter_ax(ax, x_var, y_var, title)
    
ax.legend(['Sleeplab', 'ICU'])
plt.tight_layout()

# plt.savefig(os.path.join(plots_savedir, f'Figure_CPC.jpg'), dpi=500)
# plt.savefig(os.path.join(plots_savedir, f'Figure_CPC.svg'))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [52]:
def ahi_strata_scatter_ax(ax, x_var, y_var,title):
    for i_ahi, ahi_strata_tmp in enumerate(ahi_strata):
        Xy_sleeplab = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_' + ahi_strata_tmp]==1]
        ax.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c=ahi_colors[i_ahi], s=s, alpha=alpha)
    ax.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
#     ax.legend(['Sleeplab', 'ICU'])
    ax.set_title(title)
    
    
ahi_categories = ['all', 'ahi_0_5', 'ahi_5_15', 'ahi_15_30', 'ahi_15_100', 'ahi_30_100']
ahi_names = {
    'all': 'All',
    'ahi_0_5': 'AHI < 5',
    'ahi_5_15': '5 < AHI <=15',
    'ahi_15_30': '15 < AHI <= 30',
    'ahi_15_100': 'AHI > 15',
    'ahi_30_100': 'AHI > 30',

}

ahi_strata = ['all', 'ahi_0_5', 'ahi_15_100']
ahi_colors = ['goldenrod', 'green', 'red']

In [53]:
vars_to_plot = ['cpc_hfc_lfc_ratio']
units = ['', '', '']
titles = ['CPC HFC/LFC']

fig, ax = plt.subplots(1, 1, figsize = (5, 5))
# ax = ax.flatten()
s=6
alpha = 0.5

for i_var, var_tmp in enumerate(vars_to_plot):
    
    xlabel = 'Median' + units[i_var]
    ylabel = 'IQR' + units[i_var]

    x_var = var_tmp + '_median'
    y_var = var_tmp + '_iqr'
    title = titles[i_var]
    
    ahi_strata_scatter_ax(ax, x_var, y_var, title)
    
ax.legend(['Sleeplab', 'ICU'])
plt.tight_layout()

# plt.savefig(os.path.join(plots_savedir, f'Figure_CPC.jpg'), dpi=500)
# plt.savefig(os.path.join(plots_savedir, f'Figure_CPC.svg'))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [54]:
vars_to_plot

['cpc_hfc_lfc_ratio']

In [55]:
cols = ['Median_Median', 'Median_Q25', 'Median_Q75', 'Median_IQR', 
        'IQR_Median', 'IQR_Q25', 'IQR_Q75', 'IQR_IQR',
       'Q25_Median', 'Q25_Q25', 'Q25_Q75', 'Q25_IQR',
        'Q75_Median', 'Q75_Q25', 'Q75_Q75', 'Q75_IQR',
       ]
cpc_summaries = pd.DataFrame(columns=cols)

ahi_strata = ahi_categories
var = 'cpc_hfc_lfc_ratio'



for i_ahi, ahi_strata_tmp in enumerate(ahi_strata):
    
    Xy_sleeplab = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_' + ahi_strata_tmp]==1]
    result = []
    
    for var_tmp in ['median', 'iqr', 'q25', 'q75']:

        x_var = var + '_' + var_tmp


        result.extend([Xy_sleeplab[x_var].median(),
                  Xy_sleeplab[x_var].quantile(0.25),
                  Xy_sleeplab[x_var].quantile(0.75),
                  Xy_sleeplab[x_var].quantile(0.75) - Xy_sleeplab[x_var].quantile(0.25),
               ])

    cpc_summaries.loc['sleeplab_' + ahi_strata_tmp, :] = np.array(result).flatten()

icu_versions = ['icu_4_strict', 'icu_4', 'summary_days_icu_n', 'summary_dn_icu']
for icu_version in icu_versions:

    if icu_version == 'icu_4_strict':
        Xy_icu = icu_4_strict.copy()
    elif icu_version == 'icu_4':
        Xy_icu = icu_4.copy()
    elif icu_version == 'summary_days_icu_n':
        Xy_icu = summary_days_icu_n
    elif icu_version == 'summary_dn_icu':
        Xy_icu = summary_dn_icu

    result = []
    
    for var_tmp in ['median', 'iqr', 'q25', 'q75']:

        x_var = var + '_' + var_tmp

        result.extend([Xy_icu[x_var].median(),
                  Xy_icu[x_var].quantile(0.25),
                  Xy_icu[x_var].quantile(0.75),
                  Xy_icu[x_var].quantile(0.75) - Xy_icu[x_var].quantile(0.25),
               ])

    cpc_summaries.loc['icu_' + icu_version, :] = result


cpc_summaries = cpc_summaries.astype('float')

In [56]:
cpc_summaries

,Median_Median,Median_Q25,Median_Q75,Median_IQR,IQR_Median,IQR_Q25,IQR_Q75,IQR_IQR,Q25_Median,Q25_Q25,Q25_Q75,Q25_IQR,Q75_Median,Q75_Q25,Q75_Q75,Q75_IQR
sleeplab_all,-2.387245,-3.442796,-1.113471,2.329325,3.004504,2.273472,4.022265,1.748793,-3.886096,-5.019543,-2.773217,2.246326,-0.387192,-1.856840,0.791510,2.648350
sleeplab_ahi_0_5,-2.401653,-3.313106,-1.217596,2.095510,3.140647,2.128497,4.377057,2.248560,-3.808459,-4.842995,-2.861812,1.981183,-0.078932,-1.585961,0.988266,2.574227
sleeplab_ahi_5_15,-2.193386,-3.309751,-0.896160,2.413591,3.239222,2.421479,4.333967,1.912488,-3.859339,-5.052111,-2.708716,2.343395,-0.189465,-1.358424,0.842283,2.200708
sleeplab_ahi_15_30,-2.847672,-3.569933,-1.828635,1.741298,2.696068,2.195169,3.024581,0.829412,-4.043730,-4.978056,-2.967474,2.010582,-1.023476,-2.118167,0.013922,2.132089
sleeplab_ahi_15_100,-2.940117,-3.712401,-1.581390,2.131010,2.620271,2.176165,3.043045,0.866879,-4.133180,-5.061373,-3.031908,2.029466,-1.225068,-2.582765,0.089101,2.671867
sleeplab_ahi_30_100,-3.700373,-4.777856,-2.409538,2.368319,2.476399,2.077037,3.220725,1.143688,-4.754896,-6.034587,-3.467302,2.567286,-2.329452,-3.098105,-1.579541,1.518564
icu_icu_4_strict,-1.354427,-2.546135,-0.136576,2.409559,2.170480,1.770033,2.697838,0.927805,-2.316523,-3.491280,-1.297286,2.193994,-0.001118,-1.642847,1.256484,2.899332
icu_icu_4,-1.654642,-2.554275,-0.790143,1.764132,1.930230,1.581489,2.402587,0.821097,-2.571289,-3.404617,-1.819645,1.584972,-0.695229,-1.710171,0.537632,2.247802
icu_summary_days_icu_n,-1.566265,-2.515825,-0.386799,2.129026,2.070640,1.711424,2.683137,0.971713,-2.471064,-3.423190,-1.681496,1.741694,-0.527504,-1.594970,0.881712,2.476681
icu_summary_dn_icu,-1.664293,-2.573467,-0.796140,1.777327,1.838415,1.495414,2.312194,0.816780,-2.465783,-3.400161,-1.805484,1.594677,-0.716772,-1.713004,0.377667,2.090670


In [57]:
cpc_summaries.round(2)

,Median_Median,Median_Q25,Median_Q75,Median_IQR,IQR_Median,IQR_Q25,IQR_Q75,IQR_IQR,Q25_Median,Q25_Q25,Q25_Q75,Q25_IQR,Q75_Median,Q75_Q25,Q75_Q75,Q75_IQR
sleeplab_all,-2.39,-3.44,-1.11,2.33,3.00,2.27,4.02,1.75,-3.89,-5.02,-2.77,2.25,-0.39,-1.86,0.79,2.65
sleeplab_ahi_0_5,-2.40,-3.31,-1.22,2.10,3.14,2.13,4.38,2.25,-3.81,-4.84,-2.86,1.98,-0.08,-1.59,0.99,2.57
sleeplab_ahi_5_15,-2.19,-3.31,-0.90,2.41,3.24,2.42,4.33,1.91,-3.86,-5.05,-2.71,2.34,-0.19,-1.36,0.84,2.20
sleeplab_ahi_15_30,-2.85,-3.57,-1.83,1.74,2.70,2.20,3.02,0.83,-4.04,-4.98,-2.97,2.01,-1.02,-2.12,0.01,2.13
sleeplab_ahi_15_100,-2.94,-3.71,-1.58,2.13,2.62,2.18,3.04,0.87,-4.13,-5.06,-3.03,2.03,-1.23,-2.58,0.09,2.67
sleeplab_ahi_30_100,-3.70,-4.78,-2.41,2.37,2.48,2.08,3.22,1.14,-4.75,-6.03,-3.47,2.57,-2.33,-3.10,-1.58,1.52
icu_icu_4_strict,-1.35,-2.55,-0.14,2.41,2.17,1.77,2.70,0.93,-2.32,-3.49,-1.30,2.19,-0.00,-1.64,1.26,2.90
icu_icu_4,-1.65,-2.55,-0.79,1.76,1.93,1.58,2.40,0.82,-2.57,-3.40,-1.82,1.58,-0.70,-1.71,0.54,2.25
icu_summary_days_icu_n,-1.57,-2.52,-0.39,2.13,2.07,1.71,2.68,0.97,-2.47,-3.42,-1.68,1.74,-0.53,-1.59,0.88,2.48
icu_summary_dn_icu,-1.66,-2.57,-0.80,1.78,1.84,1.50,2.31,0.82,-2.47,-3.40,-1.81,1.59,-0.72,-1.71,0.38,2.09


In [59]:
cpc_summaries[['Q25_Median', 'Q75_Median', 'Median_Median']].round(2)

,Q25_Median,Q75_Median,Median_Median
sleeplab_all,-3.89,-0.39,-2.39
sleeplab_ahi_0_5,-3.81,-0.08,-2.40
sleeplab_ahi_5_15,-3.86,-0.19,-2.19
sleeplab_ahi_15_30,-4.04,-1.02,-2.85
sleeplab_ahi_15_100,-4.13,-1.23,-2.94
sleeplab_ahi_30_100,-4.75,-2.33,-3.70
icu_icu_4_strict,-2.32,-0.00,-1.35
icu_icu_4,-2.57,-0.70,-1.65
icu_summary_days_icu_n,-2.47,-0.53,-1.57
icu_summary_dn_icu,-2.47,-0.72,-1.66


In [78]:
# histograms:


cols = ['Median_Median', 'Median_Q25', 'Median_Q75', 'Median_IQR', 
        'IQR_Median', 'IQR_Q25', 'IQR_Q75', 'IQR_IQR',
       'Q25_Median', 'Q25_Q25', 'Q25_Q75', 'Q25_IQR',
        'Q75_Median', 'Q75_Q25', 'Q75_Q75', 'Q75_IQR',
       ]
cpc_summaries = pd.DataFrame(columns=cols)

ahi_strata = ahi_categories
var = 'cpc_hfc_lfc_ratio'

ahi_categories = ['all', 'ahi_0_5', 'ahi_5_15', 'ahi_15_30', 'ahi_15_100', 'ahi_30_100']

fig, ax = plt.subplots(3,1, figsize=(3,5), sharex=True)

ahi_color = ['green', 'red']
for i_ahi, ahi_strata_tmp in enumerate(['ahi_0_5', 'ahi_15_100']):
    print(ahi_strata_tmp)
    
    Xy_sleeplab = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_' + ahi_strata_tmp]==1]


    for i_var, var_tmp in enumerate(['median', 'q25', 'q75']):

        x_var = var + '_' + var_tmp

        ax[i_var].hist(Xy_sleeplab[x_var].values, histtype='step', color=ahi_color[i_ahi], density=True)
        
        
icu_versions = ['icu_4'] # ['icu_4_strict', 'icu_4', 'summary_days_icu_n', 'summary_dn_icu']
for icu_version in icu_versions:

    if icu_version == 'icu_4_strict':
        Xy_icu = icu_4_strict.copy()
    elif icu_version == 'icu_4':
        Xy_icu = icu_4.copy()
    elif icu_version == 'summary_days_icu_n':
        Xy_icu = summary_days_icu_n
    elif icu_version == 'summary_dn_icu':
        Xy_icu = summary_dn_icu

    result = []
    
    for i_var, var_tmp in enumerate(['median', 'q25', 'q75']):

        x_var = var + '_' + var_tmp

        ax[i_var].hist(Xy_icu[x_var].values, histtype='step', color='navy', density=True)
        
ax[0].set_title('Median HFC/LFC across night')
ax[1].set_title('Q25 HFC/LFC across night')
ax[2].set_title('Q75 HFC/LFC across night')
ax[0].legend(['Sleeplab AHI<5', 'Sleeplab AHI>15', 'ICU'], bbox_to_anchor = (0.5,1.2))
plt.tight_layout()

/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/ipykernel_launcher.py:16: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  app.launch_new_instance()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

ahi_0_5
ahi_15_100
